# Service 1 — Étape 4 : évaluation du modèle fine-tuné (APRÈS) et comparaison

Exactement la même évaluation qu'à l'étape 2 (mêmes 109 questions de test, même décodage, mêmes mesures),
mais avec le modèle de base **+ l'adaptateur LoRA** entraîné à l'étape 3. La dernière cellule compare avant / après.

**Avant de lancer :** GPU T4, puis *Exécution → Tout exécuter* (environ 10 min).

## 1. Drive, code et chemins

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/cardioagent-ops"
REPO_URL = "https://github.com/yasssmine2003/CardioAgent-Ops.git"
REPO_DIR = "/content/CardioAgent-Ops"
if not os.path.isdir(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}/llm_finetuning

EVAL_DIR = f"{DRIVE}/llm/eval"
os.makedirs(EVAL_DIR, exist_ok=True)
os.environ["MLFLOW_TRACKING_URI"] = f"file:{DRIVE}/mlruns"
print("Test présent :", os.path.exists("data/final/test.jsonl"))

## 2. Installation (Unsloth, MLflow, ROUGE)
Les avertissements rouges de pip sur des conflits de versions sont habituels sur Colab.

In [ ]:
!pip install -q unsloth "mlflow>=2.17,<3" rouge_score

## 3. Chargement du modèle fine-tuné
Unsloth lit l'adaptateur, retrouve le modèle de base 4 bits qu'il complète, et charge les deux ensemble.

In [ ]:
import torch
from unsloth import FastLanguageModel

MODEL_NAME = f"{DRIVE}/llm/adapter_qwen2.5-7b-cardio"   # base 4 bits + adaptateur LoRA (étape 3)
model, tokenizer = FastLanguageModel.from_pretrained(MODEL_NAME, max_seq_length=2048, load_in_4bit=True)
FastLanguageModel.for_inference(model)                  # mode « réponse » (plus rapide)
print(f"Mémoire GPU utilisée : {torch.cuda.memory_allocated() / 1e9:.1f} Go")

## 4. Génération des réponses (sauvegarde au fur et à mesure)

In [5]:
import json
from src.evaluation import generate_answers

STAGE = "finetuned"
pred_path = f"{EVAL_DIR}/{STAGE}_predictions.jsonl"

with open("data/final/test.jsonl", encoding="utf-8") as f:
    test = [json.loads(line) for line in f]

done = []
if os.path.exists(pred_path):                        # reprise après déconnexion
    with open(pred_path, encoding="utf-8") as f:
        done = [json.loads(line) for line in f]
print(f"{len(done)} réponses déjà faites, {len(test) - len(done)} à faire")

for start in range(len(done), len(test), 10):        # par paquets de 10, sauvegardés tout de suite
    batch = generate_answers(model, tokenizer, test[start:start + 10], log_every=100)
    with open(pred_path, "a", encoding="utf-8") as f:
        for r in batch:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"{start + len(batch)}/{len(test)} sauvegardées")

0 réponses déjà faites, 109 à faire


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

10/109 sauvegardées


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

20/109 sauvegardées


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

... (13 outputs left out)

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

100/109 sauvegardées


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

109/109 sauvegardées


## 5. Scores

In [6]:
import pandas as pd
from src.evaluation import score_all

with open(pred_path, encoding="utf-8") as f:
    results = [json.loads(line) for line in f]
results, summary = score_all(results)
summary["memoire_gpu_max_go"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)

with open(f"{EVAL_DIR}/{STAGE}_scores.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
pd.Series(summary, name=STAGE).to_frame()

,finetuned
service3_json_valide,1.0000
service3_json_complet,1.0000
service3_resultat_juste,1.0000
service3_cas_limite_juste,0.8500
service3_fiabilite_juste,0.7000
service3_n,40.0000
medmcqa_bonne_lettre,0.7200
medmcqa_n,50.0000
medquad_rougeL,0.2309
medquad_n,19.0000


## 6. Quelques réponses à lire (une de chaque type)

In [7]:
for family in ["service3", "medmcqa", "medquad"]:
    r = next(x for x in results if x["family"] == family)
    print(f"===== {family} =====\nQUESTION :\n{r['question'][:500]}\n\nRÉPONSE DU MODÈLE :\n{r['prediction'][:700]}"
          f"\n\nRÉFÉRENCE :\n{r['reference'][:400]}\n\nSCORES : {r['scores']}\n")

===== service3 =====
QUESTION :
Analyse ce résultat du moteur audio (enregistrement phonocardiographique) et réponds en JSON.

Résultat du moteur audio :
{"prediction": "pathologique", "probability": 0.9148, "confidence": 0.9148, "segment_probabilities": [0.8342, 0.8873, 0.9596, 0.8969, 0.919, 0.992], "n_segments": 6, "duration_s": 34.71, "padded": false, "model": {"threshold": 0.7307}}

RÉPONSE DU MODÈLE :
{
  "resultat": "pathologique",
  "fiabilite": "élevée",
  "cas_limite": false,
  "interpretation": "Probabilité « pathologique » de 0,915, nettement au-dessus du seuil de décision (0,731). Analyse réalisée sur 6 segments de 5 s.",
  "recommandation": "Un avis cardiologique est recommandé pour confirmer ou écarter une anomalie.",
  "avertissement": "Outil d'aide au dépistage : il ne remplace pas un examen médical."
}

RÉFÉRENCE :
{
  "resultat": "pathologique",
  "fiabilite": "moyenne",
  "cas_limite": false,
  "interpretation": "Probabilité « pathologique » de 0,915, au-dessus du s

## 7. Enregistrement dans MLflow

In [8]:
import mlflow
from src.evaluation import MAX_NEW_TOKENS

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("cardioagent-llm")
with mlflow.start_run(run_name=f"eval_{STAGE}"):
    mlflow.set_tags({"stage": STAGE, "model": MODEL_NAME, "family": "evaluation"})
    mlflow.log_params({"n_test": len(results), "decoding": "greedy",
                       **{f"max_new_tokens_{k}": v for k, v in MAX_NEW_TOKENS.items()}})
    mlflow.log_metrics(summary)
    mlflow.log_artifact(pred_path)
    mlflow.log_artifact(f"{EVAL_DIR}/{STAGE}_scores.json")
print("Run MLflow enregistré :", f"eval_{STAGE}")

Run MLflow enregistré : eval_finetuned


## 8. Comparaison avant / après

In [9]:
with open(f"{EVAL_DIR}/baseline_scores.json", encoding="utf-8") as f:
    before = json.load(f)
table = pd.DataFrame({"avant (base)": before, "après (fine-tuné)": summary})
table["écart"] = table["après (fine-tuné)"] - table["avant (base)"]
table.round(3)

,avant (base),après (fine-tuné),écart
service3_json_valide,1.000,1.000,0.000
service3_json_complet,0.000,1.000,1.000
service3_resultat_juste,0.000,1.000,1.000
service3_cas_limite_juste,0.000,0.850,0.850
service3_fiabilite_juste,0.000,0.700,0.700
service3_n,40.000,40.000,0.000
medmcqa_bonne_lettre,0.560,0.720,0.160
medmcqa_n,50.000,50.000,0.000
medquad_rougeL,0.172,0.231,0.059
medquad_n,19.000,19.000,0.000
